# Tutorial 12: Lattice Reconstruction — Restore W Lattice from HLLSets

**Tier 3 — Disambiguation & Reconstruction**

When HLLSets are stored in a ring or collected from different sources,
their **lattice structure** (partial ordering by subset relation) is lost.
This tutorial shows how to **reconstruct** the W lattice using BSS morphisms.

## What You'll Learn

1. **BSS Morphism Graph** — Directed graph of pairwise BSS relationships
2. **Subset Detection** — Identifying A ⊆ B via high τ(B→A)
3. **Transitive Reduction** — Computing the Hasse diagram
4. **Level Structure** — Antichains and lattice height/width
5. **Lattice Navigation** — Chains, predecessors, successors
6. **DOT Export** — Visualizing lattice structure

## Prerequisites

| Tutorial | Concept |
|----------|---------|
| [01_hllset.ipynb](01_hllset.ipynb) | HLLSet creation, set operations |
| [07_bss.ipynb](07_bss.ipynb) | BSS (τ, ρ) directed similarity |
| [03_hll_lattice.ipynb](03_hll_lattice.ipynb) | W lattice concept |

## The Reconstruction Problem

```
Ring / Store / Network
  ┌─────────────────┐
  │  Unordered set   │
  │  of HLLSets      │     ──────▶   Reconstruct   ──────▶   W Lattice
  │  {H₁, H₂, ..}   │                                        with partial
  └─────────────────┘                                         ordering
```

**Key Insight**: BSS(B→A) ≈ 1 means A ⊆ B (A is "smaller", B "covers" A).
This gives us directed edges for the partial order.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from core.lattice_reconstruction import (
    LatticeEdge,
    LatticeLevel,
    ReconstructedLattice,
    BSSMorphismGraph,
    LatticeReconstructor,
    reconstruct_lattice,
    lattice_to_dot,
)
from core.hllset import HLLSet
from core.bss import bss

P_BITS = 10

## 1. Creating a Known Lattice

We'll build HLLSets with known subset relationships, then verify that
reconstruction recovers them.

```
          H_all (a,b,c,d,e)
         /        |        \
    H_abc        H_bcd      H_cde
    (a,b,c)     (b,c,d)    (c,d,e)
         \        |        /
          H_bc   H_cd
          (b,c)  (c,d)
              \  /
              H_c
              (c)
```

In [ ]:
# Create HLLSets with known subset structure
sets = {
    "H_c":    HLLSet.from_batch(["c"], p_bits=P_BITS),
    "H_bc":   HLLSet.from_batch(["b", "c"], p_bits=P_BITS),
    "H_cd":   HLLSet.from_batch(["c", "d"], p_bits=P_BITS),
    "H_abc":  HLLSet.from_batch(["a", "b", "c"], p_bits=P_BITS),
    "H_bcd":  HLLSet.from_batch(["b", "c", "d"], p_bits=P_BITS),
    "H_cde":  HLLSet.from_batch(["c", "d", "e"], p_bits=P_BITS),
    "H_all":  HLLSet.from_batch(["a", "b", "c", "d", "e"], p_bits=P_BITS),
}

print("HLLSets with known lattice structure:")
for name, hll in sets.items():
    print(f"  {name:<8} |H| ≈ {hll.cardinality():.0f}")

## 2. The BSS Morphism Graph

The `BSSMorphismGraph` computes pairwise BSS between all HLLSets and
classifies relationships as subset, superset, overlap, or disjoint.

In [ ]:
# Create morphism graph
morph_graph = BSSMorphismGraph(
    tau_threshold=0.9,    # τ for significant relationship
    rho_threshold=0.1,    # ρ for morphism edge
    subset_tau=0.95,      # τ threshold for A ⊆ B detection
)

# Add all nodes
for name, hll in sets.items():
    morph_graph.add_node(name, hll)

print(f"Morphism graph: {len(morph_graph.nodes)} nodes")

In [ ]:
# Compute all pairwise edges
all_edges = morph_graph.compute_all_edges()

print(f"Computed {len(all_edges)} edges")
print(f"\nEdge type breakdown:")
from collections import Counter
type_counts = Counter(e.edge_type for e in all_edges)
for edge_type, count in type_counts.most_common():
    print(f"  {edge_type:<12} {count}")

In [ ]:
# Examine subset edges — these define the partial order
subset_edges = morph_graph.get_subset_edges()

print(f"\nSubset edges (A ⊂ B):")
for edge in subset_edges:
    print(f"  {edge.source_id:<8} ⊂ {edge.target_id:<8}  τ_fwd={edge.tau_forward:.3f}")

In [ ]:
# Graph statistics
stats = morph_graph.stats()

print(f"Morphism graph statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 3. Understanding Edge Classification

An edge between A and B is classified based on BSS in both directions:

| BSS(A→B) τ | BSS(B→A) τ | Classification |
|-------------|-------------|----------------|
| High | High | **Equal** (A ≈ B) |
| Low | High | **Subset** (A ⊂ B) |
| High | Low | **Superset** (A ⊃ B) |
| Medium | Medium | **Overlap** |
| Low | Low | **Disjoint** (no edge) |

In [ ]:
# Examine a specific edge in detail
edge = morph_graph.compute_edge("H_bc", "H_abc")

if edge:
    print(f"Edge: {edge.source_id} → {edge.target_id}")
    print(f"  BSS forward  (H_bc→H_abc):  τ={edge.tau_forward:.3f}, ρ={edge.rho_forward:.3f}")
    print(f"  BSS backward (H_abc→H_bc):  τ={edge.tau_backward:.3f}, ρ={edge.rho_backward:.3f}")
    print(f"  is_subset:   {edge.is_subset}   (H_bc ⊂ H_abc)")
    print(f"  is_superset: {edge.is_superset}")
    print(f"  edge_type:   {edge.edge_type}")
    print(f"  is_comparable: {edge.is_comparable}")

## 4. Lattice Reconstruction (One-Liner)

The `reconstruct_lattice` convenience function performs all steps:
compute edges → extract subset order → transitive reduction → level assignment.

In [ ]:
# Reconstruct lattice from HLLSets
hllsets = list(sets.values())
node_ids = list(sets.keys())

lattice = reconstruct_lattice(
    hllsets,
    node_ids=node_ids,
    tau_threshold=0.9,
    subset_tau=0.95,
)

print(f"Reconstructed lattice:")
print(f"  {lattice}")
print(f"\n  Nodes:       {lattice.num_nodes}")
print(f"  All edges:   {lattice.num_edges}")
print(f"  Hasse edges: {lattice.num_hasse_edges}")
print(f"  Levels:      {lattice.num_levels}")
print(f"  Height:      {lattice.height}")
print(f"  Width:       {lattice.width}")
print(f"  Is lattice:  {lattice.is_lattice}")

## 5. Level Structure (Antichains)

Each level contains elements that are **incomparable** — neither is a subset
of the other. Level 0 is the bottom (most specific), higher levels are more general.

In [ ]:
# Examine levels
print("Level structure:")
for level in lattice.levels:
    print(f"\n  Level {level.level} ({len(level)} nodes):")
    print(f"    Cardinality range: [{level.cardinality_range[0]:.0f}, {level.cardinality_range[1]:.0f}]")
    for nid in level.node_ids:
        card = lattice.nodes[nid].cardinality()
        lvl = lattice.get_level(nid)
        print(f"    {nid:<8} |H|≈{card:.0f}  (level={lvl})")

In [ ]:
# Bottom and top elements
print(f"Bottom elements (minimal): {lattice.bottom_ids}")
print(f"Top elements (maximal):    {lattice.top_ids}")

## 6. Hasse Diagram Edges

The Hasse diagram keeps only **covering relations**: A → B means A ⊂ B
and there is no C with A ⊂ C ⊂ B. This is the **transitive reduction**
of the full subset ordering.

In [ ]:
# Hasse edges (covering relations only)
print(f"Hasse diagram edges ({lattice.num_hasse_edges} total):")
for edge in lattice.hasse_edges:
    src_level = lattice.get_level(edge.source_id)
    tgt_level = lattice.get_level(edge.target_id)
    print(
        f"  {edge.source_id:<8} (lvl {src_level}) "
        f"→ {edge.target_id:<8} (lvl {tgt_level})  "
        f"τ={edge.tau_forward:.3f}"
    )

In [ ]:
# Compare: full subset edges vs Hasse edges (transitive reduction)
full_subset = [e for e in lattice.edges if e.is_subset]
print(f"Full subset edges:       {len(full_subset)}")
print(f"Hasse edges (reduced):   {lattice.num_hasse_edges}")
print(f"Removed by reduction:    {len(full_subset) - lattice.num_hasse_edges}")

# The removed edges are the transitive ones (e.g., H_c ⊂ H_all is implied
# by H_c ⊂ H_bc and H_bc ⊂ H_abc and H_abc ⊂ H_all)

## 7. Lattice Navigation

Navigate the lattice: find predecessors, successors, and chains.

In [ ]:
# Predecessors and successors of a node
node = "H_bcd"
preds = lattice.predecessors(node)
succs = lattice.successors(node)

print(f"Node: {node}")
print(f"  Predecessors (covered by {node}): {preds}")
print(f"  Successors   (cover {node}):      {succs}")

In [ ]:
# Chains: maximal paths from a node to top/bottom
bottom_node = lattice.bottom_ids[0] if lattice.bottom_ids else None

if bottom_node:
    chain_up = lattice.chain_to_top(bottom_node)
    print(f"Chain from {bottom_node} to top:")
    print(f"  {' → '.join(chain_up)}")

    top_node = lattice.top_ids[0] if lattice.top_ids else None
    if top_node:
        chain_down = lattice.chain_to_bottom(top_node)
        print(f"\nChain from {top_node} to bottom:")
        print(f"  {' → '.join(chain_down)}")

## 8. DOT Visualization

Export the lattice to DOT format for Graphviz rendering.
The diagram flows bottom-to-top, with nodes grouped by level.

In [ ]:
# Generate DOT representation
dot = lattice_to_dot(lattice)
print(dot)

## 9. Step-by-Step Reconstruction with LatticeReconstructor

For finer control, use `LatticeReconstructor` directly.
This lets you inspect intermediate state — the morphism graph before reduction.

In [ ]:
# Step-by-step reconstruction
reconstructor = LatticeReconstructor(
    tau_threshold=0.9,
    subset_tau=0.95,
)

# Step 1: Add HLLSets
ids = reconstructor.add_hllsets(hllsets, node_ids)
print(f"Step 1 — Added {len(ids)} nodes:")
for nid in ids:
    print(f"  {nid}")

In [ ]:
# Step 2: Inspect morphism graph before reconstruction
graph = reconstructor.graph
print(f"\nStep 2 — Morphism graph before reconstruction:")
print(f"  Nodes: {len(graph.nodes)}")
print(f"  Edges: {len(graph.edges)} (before compute_all_edges)")

# Compute a single edge manually
single_edge = graph.compute_edge("H_c", "H_all")
if single_edge:
    print(f"\n  Manual edge H_c → H_all:")
    print(f"    type: {single_edge.edge_type}")
    print(f"    is_subset: {single_edge.is_subset}")

In [ ]:
# Step 3: Full reconstruction
lattice2 = reconstructor.reconstruct()

print(f"\nStep 3 — Reconstructed lattice:")
print(f"  {lattice2}")
print(f"\nStats:")
for key, value in lattice2.stats.items():
    print(f"  {key}: {value}")

## 10. Practical Example: Document Corpus Lattice

Reconstruct a lattice from documents with varying topic overlap.

In [ ]:
# Documents with overlapping topics
documents = {
    "ml_basics":    ["machine", "learning", "data", "model"],
    "deep_learn":   ["machine", "learning", "data", "model", "deep", "neural", "network"],
    "nlp":          ["language", "model", "text", "data", "neural", "network"],
    "full_ai":      ["machine", "learning", "data", "model", "deep", "neural", "network",
                     "language", "text", "reinforcement"],
    "data_core":    ["data", "model"],
}

doc_hllsets = {name: HLLSet.from_batch(tokens, p_bits=P_BITS)
               for name, tokens in documents.items()}

print("Document corpus:")
for name, hll in doc_hllsets.items():
    print(f"  {name:<15} |H|≈{hll.cardinality():.0f}  tokens: {documents[name]}")

In [ ]:
# Reconstruct lattice
doc_lattice = reconstruct_lattice(
    list(doc_hllsets.values()),
    node_ids=list(doc_hllsets.keys()),
    tau_threshold=0.9,
    subset_tau=0.95,
)

print(f"Document lattice: {doc_lattice}")
print(f"\nLevel structure:")
for level in doc_lattice.levels:
    nodes_str = ", ".join(doc_lattice.levels[level.level].node_ids)
    print(f"  Level {level.level}: [{nodes_str}]")

In [ ]:
# Hasse edges show containment relationships
print(f"\nContainment relations (Hasse edges):")
for edge in doc_lattice.hasse_edges:
    src_card = doc_lattice.nodes[edge.source_id].cardinality()
    tgt_card = doc_lattice.nodes[edge.target_id].cardinality()
    print(f"  {edge.source_id:<15} (|{src_card:.0f}|) ⊂ {edge.target_id:<15} (|{tgt_card:.0f}|)")

In [ ]:
# DOT for document lattice
doc_dot = lattice_to_dot(doc_lattice)
print(doc_dot)

## 11. Lattice Properties

Examine key properties of the reconstructed lattice.

In [ ]:
# Height and width characterize lattice shape
print(f"Lattice properties:")
print(f"  Height (longest chain - 1): {doc_lattice.height}")
print(f"  Width  (largest antichain): {doc_lattice.width}")
print(f"  Is lattice: {doc_lattice.is_lattice}")

# Enumerate all chains from bottom to top
print(f"\nMaximal chains from each bottom element:")
for bot in doc_lattice.bottom_ids:
    chain = doc_lattice.chain_to_top(bot)
    print(f"  {' → '.join(chain)}")

## 12. Tuning Thresholds

The `subset_tau` threshold controls how strict subset detection is.
Lower values find more (possibly approximate) subset relations.

In [ ]:
# Compare different subset_tau thresholds
for tau in [0.99, 0.95, 0.85, 0.75]:
    lat = reconstruct_lattice(
        list(doc_hllsets.values()),
        node_ids=list(doc_hllsets.keys()),
        subset_tau=tau,
    )
    print(f"subset_τ={tau:.2f}: {lat.num_hasse_edges} Hasse edges, {lat.num_levels} levels, height={lat.height}")

## 13. Lattice Statistics Summary

In [ ]:
# Comprehensive statistics for the document lattice
print("Comprehensive lattice statistics:")
print(f"  Nodes:         {doc_lattice.num_nodes}")
print(f"  All edges:     {doc_lattice.num_edges}")
print(f"  Hasse edges:   {doc_lattice.num_hasse_edges}")
print(f"  Levels:        {doc_lattice.num_levels}")
print(f"  Height:        {doc_lattice.height}")
print(f"  Width:         {doc_lattice.width}")
print(f"  Bottom:        {doc_lattice.bottom_ids}")
print(f"  Top:           {doc_lattice.top_ids}")
print(f"\nStats dict:")
for key, value in doc_lattice.stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

## Summary

In this tutorial, you learned:

1. **BSS Morphism Graph** — Pairwise BSS classifies edges as subset/superset/overlap
2. **Subset Detection** — BSS(B→A) ≈ 1 implies A ⊆ B
3. **Transitive Reduction** — Removes implied edges to yield the Hasse diagram
4. **Level Structure** — Antichains of incomparable elements
5. **Navigation** — Chains, predecessors, successors through the partial order

### API Reference

| Class/Function | Purpose |
|----------------|---------|
| `BSSMorphismGraph` | Low-level graph with edge computation |
| `LatticeReconstructor` | Step-by-step reconstruction |
| `reconstruct_lattice()` | One-liner convenience function |
| `lattice_to_dot()` | DOT export for visualization |

### ReconstructedLattice Properties

| Property | Description |
|----------|-------------|
| `height` | Length of longest chain − 1 |
| `width` | Size of largest antichain (level) |
| `levels` | List of `LatticeLevel` antichains |
| `hasse_edges` | Covering relations (transitive reduction) |
| `bottom_ids` / `top_ids` | Minimal / maximal elements |

### The Reconstruction Algorithm

```
1. Compute pairwise BSS for all HLLSets          O(n²)
2. Classify edges: subset / superset / overlap
3. Extract subset partial order
4. Compute transitive reduction (Hasse diagram)
5. Topological sort → level assignment
6. Identify bottom / top elements
```

### Connection to Framework

- **BSS** (Tutorial 07) provides the directed similarity metric
- **HLLSet De Bruijn** (Tutorials 09, 11) recovers evolution *order*
- **Lattice Reconstruction** recovers the partial *order* (containment)
- Together: full structural recovery from unordered HLLSet collections

### Next Steps

- **Tutorial 13**: Noether — Conservation laws and steering